## Case Study Assignement (word for word copy)
You are an employee of the fictitious company “202”, which sells several individual parts to engine manufacturers. The engine manufacturers supply the automobile brands “OEM1” and “OEM2”. The marketing department asks you to determine the market share for each part that the company produces compared to the competition. For advertising-technical reasons you are to determine also, in how many engines parts of “202” are installed. It is planned to advertise with the slogan: “In every xth engine there are parts from 202”. Furthermore, the high quality of the parts is to be advertised. Therefore, you are to determine the relative defect frequency of all parts in comparison to the competition. For the second part of your analysis, consider only engines that were installed in vehicles of the car brand “OEM1”. The time range in which the parts considered in the analysis were produced should be freely selectable.
## Main Case Study Task's (categorized in 4 points)
* 1. What is the market share of our company202 parts vs the competition?
* 2. Compare the defect frequency of our parts vs the competition for OEM 1 cars
* 3. How many Engines have parts from our company build in?
* 4. Create a Dashboard visualization

## The Plan is to gain an understanding of ...
* what kind of data and how much data we have?
* how this data is related?
* what role does our company 202 play in that data?
* what data is important for our case study?

## The Data selection strategy
* perfoming manual lookups into the files
* find out how much data we deal with
* how the data is formated
* gain understanding about what the columns mean and how they are correlated, by manually looking at the files and column names
* get an idea about what data could/would be important for the analysis
* importing the files
* identify relevant files through code
* visualize and document findings properly for better understating about how the relevant files are interesting in our case study

## Data structure
* The data is structured in 6 folders.

```
data/
├── Einzelteil/
├── Fahrzeug/
├── Geodaten/
├── Komponente/
├── Logistikverzug/
└── Zulassungen/
```
For language consistency we will refer to these folders and files in english.

![translation](translation.png)

First about single parts also refered as parts from here on.

#### We performed a manual lookup and these were our findings:
* lookups done with ```less <filename>``` in Git bash because the files have a big file size (100MB+/file)

**Parts Lookup**:
* There are a mix of txt and csv files. Some of the txt files are formatted terribly
* Looking at a better formated file, we can see following columns"
"";"X1";"ID_T04";"Herstellernummer";"Werksnummer";"Fehlerhaft";"Fehlerhaft_Datum";"Fehlerhaft_Fahrleistung";"Produktionsdatum_Origin_01011970";"origin"
* The part ID (for example: "4-204-2042-23") together with the other columns gives an idea on how the id is structured.
* The first number stands for the part type, in this case part type 4
* The second number stands for the Tier-2 manufacturer number, here I can also find the manufacturer number 202 which seems to be us
* the third number stands for the Tier-2 plant number. A manufacturer can have multiple plants
* the final number seems to be a number that got added that makes the id unique
* Other than that we get to know if a certain part is faulty and when it was produced, tho the date is in a weird format using default time 01.01.1970 + offset
* In total there are 38 part types, part type 1-40. For some reason T28,T29 seems to be missing

**Components Lookup**:
* There are 2 file compositions
* Bestandteile_Komponente files shows the parts id's that build this component
* Komponente files show similar informations as the part files just with components
* We see which manufacturer made it, in which plant, at which date and if its faulty
* This time the manufactuere number and plant numbers start with 1. Probably because they are considered a T1 supplier
* The Component id (for example: "K1BE1-101-1011-11")is also interesting to look at because it also contains a lot of information actually
* the first part stands for the component type
* the second part stands for the manufacturer number
* the third part stands for the plant number 
* the last part stands probably again for a number that makes this id unique

**Car Lookup**:
* From the file names there seems to be 2 OEM manufacturers who produce each 2 type of cars
* Bestandteile Fahrzeuge contains 4 component id's that make up the car with a car id.
* Looking through multiple files it gives more information about the component files
* K references the component. There seems to be components from K1-K7, from the car file column names we can find out what each K represents
* K1 - Engine, K2 - Seats, K3 - Transmission, K4-K7 - Body. I was wondering why the body uses multiple K Numbers. After more lookup it became clear that each Car model type uses a specific body
* Going from the assumption that Everything are German abreviations we could interpret the abbreviations as following:
* BE=Benzin (petrol), DI=Diesel, ST=Stoff (fabric), LE=Leder (leather), SG=Schaltgetriebe (manual), AG=Automatikgetriebe (automatic).
* The other type of file is the Fahrzeuge OEM files. They say the manufacturing number, plant number of the OEM and some faulty analysis columns etc.

**Geodata Lookup**
* seems less useful for this case study, it describes where the plants of each manufacturer are located at

**Logistikverzug Lookup**
* The component K7 is a duplicate from the components folder and the delay shows informaton about the id, if its faulty or not and when it arrived at the oem.

**Zulassungen**
* it shows the German "Zulassungen" of all cars and where/when they are registered
* also doesnt seem important for this case study


### What data to check out
* From the question we can say that we produce parts for engines
* it is not guaranteed and should be checked to be safe
* What parts do we produce and what components are made out of our parts
* The component and part files seems most important. AS they also contain a column if its faulty they can be usefull for faulty analysis later on aswell
* The case analysis wants us to focus on the parts and not the components. Komponente files and the faulty rates of components are not needed here then. So we will only need the Bestandteile_Komponente files
* The case study want us to be able to differenciate between supplieng the oem1 and oem2 manufacturers. 
* We will need Bestandteile_Fahrzeuge_OEMX_Typxx files aswell to make that disticting
* Due to the structure and the use of ids and how they are used as a relational reference to other tables this whole data is like a relational database.

## Explorative Data Analysis Plan
* import and analyse part, component and oem files
* to reduce the import time and efforts I did filter the part files manually, to not needing to import 38 part files


* by using CONTOL+F we manually searched the 38 files by the manufacturing number "-202-" or "202"
* We found that only T01-T05 contain parts from manufacturer 202

## Importing revelant part files and tidy up data and datatypes


In [3]:
import io
from pathlib import Path
import pandas as pd

# Set path, filenames, field and row seperator
DATA = Path("data/Einzelteil")

FILES = {
    "T01": ("Einzelteil_T01.txt", b" | | ", b" "),
    "T02": ("Einzelteil_T02.txt", b"  ", b"\t"),
    "T03": ("Einzelteil_T03.txt", b"|", b"\x0b"),
    "T04": ("Einzelteil_T04.csv", b";", b"\n"),
    "T05": ("Einzelteil_T05.csv", b",", b"\n"),
}

def load(key: str) -> pd.DataFrame:
    filename, field_sep, row_sep = FILES[key]
    
    # Replacing field- and row- seperators in memory
    raw = (DATA / filename).read_bytes().replace(field_sep, b"\x01").replace(row_sep, b"\n")
    
    # Importing memory data in the dataframe
    df = pd.read_csv(io.BytesIO(raw), sep="\x01", na_values=["NA"], dtype=str)
    
    # Combine same columns (.x, .y)
    for col in set(c.removesuffix(".x").removesuffix(".y") for c in df.columns):
        # Only do it if these columns exist
        if col + ".x" in df.columns and col + ".y" in df.columns:
            # Combine columns
            df[col] = df[col + ".x"].combine_first(df[col + ".y"])

    # Standartize id and time columns
    part = key.upper()
    df["Part_ID"] = df.get(f"ID_{part}", df.get("Part_ID"))

    
    if "Produktionsdatum" not in df.columns and "Produktionsdatum_Origin_01011970" in df.columns:
        
        # Calculating productiondate if it is not there
        df["Produktionsdatum"] = pd.to_datetime("1970-01-01") + pd.to_timedelta(
            df["Produktionsdatum_Origin_01011970"].astype(float), unit="D"
        )

    # Convert datatypes
    df["Produktionsdatum"] = pd.to_datetime(df["Produktionsdatum"])
    df["Fehlerhaft_Datum"] = pd.to_datetime(df["Fehlerhaft_Datum"])
    df["Fehlerhaft_Fahrleistung"] = df["Fehlerhaft_Fahrleistung"].str.replace(",", ".", regex=False).astype(float)
    
    int_cols = ["Herstellernummer", "Werksnummer", "Fehlerhaft"]
    df[int_cols] = df[int_cols].astype("Int64")

    # Columns to keep
    columns = [
        "Part_ID", 
        "Herstellernummer", 
        "Werksnummer", 
        "Produktionsdatum", 
        "Fehlerhaft", 
        "Fehlerhaft_Datum", 
        "Fehlerhaft_Fahrleistung"
    ]

    return df[columns].reset_index(drop=True)

In [4]:
part_names = ["T01", "T02", "T03", "T04", "T05"]
part_files = []
for part in part_names:
    part_files.append(load(part))

In [5]:
for part in part_files:
    display(part.head(5))

,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,1-201-2011-247,201,2011,2008-11-07,0,NaT,0.0
1,1-201-2011-429,201,2011,2008-11-07,0,NaT,0.0
2,1-201-2011-363,201,2011,2008-11-07,1,2009-09-30,12983.0
3,1-201-2011-30,201,2011,2008-11-07,0,NaT,0.0
4,1-201-2011-72,201,2011,2008-11-07,1,2009-09-30,12983.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,2-201-2011-239,201,2011,2008-11-07,1,2010-04-09,38354.158904
1,2-201-2011-304,201,2011,2008-11-07,0,NaT,0.000000
2,2-201-2011-125,201,2011,2008-11-07,1,2010-04-09,38354.158904
3,2-201-2011-55,201,2011,2008-11-07,0,NaT,0.000000
4,2-201-2011-133,201,2011,2008-11-07,0,NaT,0.000000


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,3-202-2023-249,202,2023,2008-11-07,0,NaT,0.0
1,3-202-2022-8,202,2022,2008-11-07,0,NaT,0.0
2,3-202-2023-192,202,2023,2008-11-07,0,NaT,0.0
3,3-202-2023-16,202,2023,2008-11-07,0,NaT,0.0
4,3-202-2023-258,202,2023,2008-11-07,0,NaT,0.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,4-204-2043-113,204,2043,2008-11-07,0,NaT,0.0
1,4-202-2023-18,202,2023,2008-11-07,0,NaT,0.0
2,4-204-2043-98,204,2043,2008-11-07,0,NaT,0.0
3,4-202-2023-51,202,2023,2008-11-07,0,NaT,0.0
4,4-204-2043-169,204,2043,2008-11-07,0,NaT,0.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,5-201-2012-82,201,2012,2008-11-07,1,2010-07-10,43163.430137
1,5-201-2012-172,201,2012,2008-11-07,0,NaT,0.000000
2,5-201-2012-23,201,2012,2008-11-07,0,NaT,0.000000
3,5-201-2012-47,201,2012,2008-11-07,1,2010-07-10,43163.430137
4,5-201-2012-101,201,2012,2008-11-07,0,NaT,0.000000


In [163]:
import io
from pathlib import Path
import pandas as pd

DATA = Path("data/Einzelteil")

FILES = {
    "t01": ("Einzelteil_T01.txt", b" | | ", b" "),
    "t02": ("Einzelteil_T02.txt", b"  ", b"\t"),
    "t03": ("Einzelteil_T03.txt", b"|", b"\x0b"),
    "t04": ("Einzelteil_T04.csv", b";", b"\n"),
    "t05": ("Einzelteil_T05.csv", b",", b"\n"),
}

# Bytes, die beim Teilimport vom Dateianfang gelesen werden.
# Muss groß genug sein, damit nrows+1 vollstaendige Zeilen enthalten sind.
CHUNK_BYTES = 2_000_000


def load(key: str, nrows: int | None = 1000) -> pd.DataFrame:
    filename, field_sep, row_sep = FILES[key]
    path = DATA / filename

    # Byte vodoo (replacing field- and row- seperators)
    if nrows is None:
        raw = path.read_bytes()
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
    else:
        with open(path, "rb") as f:
            raw = f.read(CHUNK_BYTES)
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
        lines = raw.split(b"\n")
        if len(lines) < nrows + 2 and path.stat().st_size > CHUNK_BYTES:
            raise ValueError(
                f"{filename}: {CHUNK_BYTES} Bytes reichen fuer {nrows} Zeilen nicht aus, "
                "CHUNK_BYTES erhoehen"
            )
        # letzte, evtl. abgeschnittene Zeile faellt raus
        raw = b"\n".join(lines[: nrows + 1])

    df = pd.read_csv(io.BytesIO(raw), sep="\x01", na_values=["NA"], dtype=str)

    # Combine same columns (.x, .y)
    for col in set(c.removesuffix(".x").removesuffix(".y") for c in df.columns):
        if col + ".x" in df.columns and col + ".y" in df.columns:
            df[col] = df[col + ".x"].combine_first(df[col + ".y"])

    # Standartize id and time columns
    part = key.upper()
    df["Part_ID"] = df.get(f"ID_{part}", df.get("Part_ID"))

    if "Produktionsdatum" not in df.columns and "Produktionsdatum_Origin_01011970" in df.columns:
        df["Produktionsdatum"] = pd.to_datetime("1970-01-01") + pd.to_timedelta(
            df["Produktionsdatum_Origin_01011970"].astype(float), unit="D"
        )

    # Convert datatypes
    df["Produktionsdatum"] = pd.to_datetime(df["Produktionsdatum"])
    df["Fehlerhaft_Datum"] = pd.to_datetime(df["Fehlerhaft_Datum"])
    df["Fehlerhaft_Fahrleistung"] = df["Fehlerhaft_Fahrleistung"].str.replace(",", ".", regex=False).astype(float)

    int_cols = ["Herstellernummer", "Werksnummer", "Fehlerhaft"]
    df[int_cols] = df[int_cols].astype("Int64")

    columns = [
        "Part_ID",
        "Herstellernummer",
        "Werksnummer",
        "Produktionsdatum",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]

    return df[columns].reset_index(drop=True)

Importing Einzelteile files

In [21]:
# Loading the part files by using our helper function
t01 = load("T01")
t02 = load("T02")
t03 = load("T03")
t04 = load("T04")
t05 = load("T05")

#### Combining Data

In [20]:
part_files_filtered = []

# Filter relevant columns
for i, part in enumerate(part_files):
    part["Part_Type"] = part_names[i]
    part_files_filtered.append(
        part[["Part_ID", "Herstellernummer", "Produktionsdatum", "Fehlerhaft", "Part_Type"]]
    )

# Merge on row
einzelteile_zusammen = pd.concat(part_files_filtered,axis=0, ignore_index=True)
part_type_column = einzelteile_zusammen.pop("Part_Type")
einzelteile_zusammen.insert(0, "Part_Type",part_type_column)

# Remove rows with only NAN Data
einzelteile_zusammen = einzelteile_zusammen.dropna(how="all")

# Display information
display(t01.tail(), einzelteile_zusammen.tail(), t04.tail())
display(t05.isnull().sum())

,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
2563279,1-202-2021-1278613,202,2021,2016-11-04,1,2018-06-05,35241.16
2563280,1-202-2021-1278624,202,2021,2016-11-04,0,NaT,0.00
2563281,1-202-2021-1278568,202,2021,2016-11-04,0,NaT,0.00
2563282,1-202-2021-1278619,202,2021,2016-11-04,0,NaT,0.00
2563283,1-202-2021-1278588,202,2021,2016-11-04,0,NaT,0.00


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft
9986093,T05,5-202-2012-593271,202,2016-10-24,0
9986094,T05,5-202-2012-592854,202,2016-10-22,0
9986095,T05,5-202-2012-593371,202,2016-10-24,1
9986096,T05,5-202-2012-595161,202,2016-10-29,0
9986097,T05,5-202-2012-593437,202,2016-10-29,0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
1192625,4-204-2042-140191,204,2042,2016-11-05,0,NaT,0.0
1192626,4-204-2042-140192,204,2042,2016-11-05,0,NaT,0.0
1192627,4-204-2042-140177,204,2042,2016-11-05,0,NaT,0.0
1192628,4-204-2042-140158,204,2042,2016-11-04,0,NaT,0.0
1192629,4-204-2042-140174,204,2042,2016-11-05,0,NaT,0.0


Part_ID                          0
Herstellernummer                 0
Werksnummer                      0
Produktionsdatum                 0
Fehlerhaft                       0
Fehlerhaft_Datum           1073826
Fehlerhaft_Fahrleistung          0
dtype: int64

importing komponente data

In [22]:
file_paths = [
    'data/Komponente/Bestandteile_Komponente_K1BE1.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI1.csv',
    'data/Komponente/Bestandteile_Komponente_K1BE2.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI2.csv'
]

engine_dfs = []

# for path in file_paths:
#     df = pd.read_csv(path, sep=';').drop(columns=['Unnamed: 0'])
#     engine_dfs.append(df)


komponente_k1be1 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1BE1.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1di1 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1DI1.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1be2 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1BE2.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1di2 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1DI2.csv", sep=';').drop(columns=['Unnamed: 0'])




#removing the not needed parts

komponente_k1di1 = komponente_k1di1[["ID_T1", "ID_T2","ID_T5", "ID_K1DI1"]]
komponente_k1be2 = komponente_k1be2[["ID_T1", "ID_T2", "ID_K1BE2"]]
komponente_k1di2 = komponente_k1di2[["ID_T1","ID_T2","ID_K1DI2"]]

display(komponente_k1be1.head())
display(komponente_k1di1.head())
display(komponente_k1be2.head())
display(komponente_k1di2.head())

# Check for bad Data or incomplete Data
display(komponente_k1be1.isna().sum())
display(komponente_k1di1.isna().sum())
display(komponente_k1be2.isna().sum())
display(komponente_k1di2.isna().sum())

,ID_T1,ID_T2,ID_T3,ID_T4,ID_K1BE1
0,1-201-2011-45,2-201-2011-161,3-202-2023-14,4-202-2023-20,K1BE1-101-1011-1
1,1-201-2011-429,2-201-2011-239,3-202-2023-16,4-202-2023-51,K1BE1-101-1011-2
2,1-201-2011-399,2-201-2011-220,3-202-2023-46,4-202-2023-93,K1BE1-101-1011-3
3,1-201-2011-335,2-202-2022-463,3-202-2023-149,4-204-2042-18,K1BE1-101-1011-4
4,1-204-2044-188,2-202-2022-675,3-202-2023-152,4-204-2042-40,K1BE1-101-1011-5


,ID_T1,ID_T2,ID_T5,ID_K1DI1
0,1-204-2044-27,2-201-2011-144,5-202-2012-89,K1DI1-101-1041-1
1,1-202-2021-32,2-202-2022-577,5-202-2012-155,K1DI1-101-1041-2
2,1-201-2011-238,2-202-2022-514,5-202-2012-199,K1DI1-101-1041-3
3,1-202-2021-297,2-202-2022-626,5-201-2012-57,K1DI1-101-1041-4
4,1-204-2044-73,2-201-2011-209,5-201-2012-157,K1DI1-101-1041-5


,ID_T1,ID_T2,ID_K1BE2
0,1-202-2021-339,2-202-2022-171,K1BE2-101-1011-1
1,1-201-2011-57,2-201-2011-260,K1BE2-101-1011-2
2,1-202-2021-101,2-202-2022-335,K1BE2-101-1011-3
3,1-204-2044-201,2-201-2011-312,K1BE2-101-1011-4
4,1-201-2011-26,2-202-2022-987,K1BE2-101-1011-5


,ID_T1,ID_T2,ID_K1DI2
0,1-204-2044-183,2-202-2022-551,K1DI2-103-1031-1
1,1-201-2011-246,2-202-2022-738,K1DI2-103-1031-2
2,1-201-2011-152,2-201-2011-181,K1DI2-103-1031-3
3,1-201-2011-134,2-202-2022-561,K1DI2-103-1031-4
4,1-202-2021-437,2-202-2022-724,K1DI2-103-1031-5


ID_T1       0
ID_T2       0
ID_T3       0
ID_T4       0
ID_K1BE1    0
dtype: int64

ID_T1       0
ID_T2       0
ID_T5       0
ID_K1DI1    0
dtype: int64

ID_T1       0
ID_T2       0
ID_K1BE2    0
dtype: int64

ID_T1       0
ID_T2       0
ID_K1DI2    0
dtype: int64

Melting the data

In [24]:
#K1BE1
melted_k1be1 = komponente_k1be1.melt(id_vars="ID_K1BE1", var_name="Part_Type", value_name="Part_ID")
melted_k1be1 = melted_k1be1.sort_values(["ID_K1BE1", "Part_Type"])
melted_k1be1 = melted_k1be1.drop(columns=["Part_Type"])
display(melted_k1be1.isna().sum())

#K1DI1
melted_k1di1 = komponente_k1di1.melt(id_vars="ID_K1DI1", var_name="Part_Type", value_name="Part_ID")
melted_k1di1 = melted_k1di1.sort_values(["ID_K1DI1", "Part_Type"])
melted_k1di1 = melted_k1di1.drop(columns=["Part_Type"])
display(melted_k1di1.head())

#K1BE2
melted_k1be2 = komponente_k1be2.melt(id_vars="ID_K1BE2", var_name="Part_Type", value_name="Part_ID")
melted_k1be2 = melted_k1be2.sort_values(["ID_K1BE2", "Part_Type"])
melted_k1be2 = melted_k1be2.drop(columns=["Part_Type"])
display(melted_k1be2.head())

#K1DI2
melted_k1di2 = komponente_k1di2.melt(id_vars="ID_K1DI2", var_name="Part_Type", value_name="Part_ID")
melted_k1di2 = melted_k1di2.sort_values(["ID_K1DI2", "Part_Type"])
melted_k1di2 = melted_k1di2.drop(columns=["Part_Type"])
display(melted_k1di2.head())


ID_K1BE1    0
Part_ID     0
dtype: int64

,ID_K1DI1,Part_ID
0,K1DI1-101-1041-1,1-204-2044-27
1192630,K1DI1-101-1041-1,2-201-2011-144
2385260,K1DI1-101-1041-1,5-202-2012-89
9,K1DI1-101-1041-10,1-201-2011-76
1192639,K1DI1-101-1041-10,2-202-2022-219


,ID_K1BE2,Part_ID
0,K1BE2-101-1011-1,1-202-2021-339
409422,K1BE2-101-1011-1,2-202-2022-171
9,K1BE2-101-1011-10,1-202-2021-71
409431,K1BE2-101-1011-10,2-202-2022-145
435,K1BE2-101-1011-100,1-201-2011-1379


,ID_K1DI2,Part_ID
56,K1DI2-102-1021-1,1-201-2011-352
409478,K1DI2-102-1021-1,2-202-2022-1051
65,K1DI2-102-1021-10,1-202-2021-597
409487,K1DI2-102-1021-10,2-202-2022-1400
211,K1DI2-102-1021-100,1-202-2021-1049


In [25]:

melted_k1be1["Engine Type"] = "K1BE1"

melted_k1di1["Engine Type"] = "K1DI1"

melted_k1be2["Engine Type"] = "K1BE2"

melted_k1di2["Engine Type"] = "K1DI2"

display(einzelteile_zusammen.head(), melted_k1be1.head(), melted_k1di1.head(), melted_k1be2.head(), melted_k1di2.head())
display(einzelteile_zusammen.isna().sum(), melted_k1be1.isna().sum(), melted_k1di1.isna().sum(), melted_k1be2.isna().sum(), melted_k1di2.isna().sum())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft
0,T01,1-201-2011-247,201,2008-11-07,0
1,T01,1-201-2011-429,201,2008-11-07,0
2,T01,1-201-2011-363,201,2008-11-07,1
3,T01,1-201-2011-30,201,2008-11-07,0
4,T01,1-201-2011-72,201,2008-11-07,1


,ID_K1BE1,Part_ID,Engine Type
0,K1BE1-101-1011-1,1-201-2011-45,K1BE1
1192630,K1BE1-101-1011-1,2-201-2011-161,K1BE1
2385260,K1BE1-101-1011-1,3-202-2023-14,K1BE1
3577890,K1BE1-101-1011-1,4-202-2023-20,K1BE1
9,K1BE1-101-1011-10,1-201-2011-37,K1BE1


,ID_K1DI1,Part_ID,Engine Type
0,K1DI1-101-1041-1,1-204-2044-27,K1DI1
1192630,K1DI1-101-1041-1,2-201-2011-144,K1DI1
2385260,K1DI1-101-1041-1,5-202-2012-89,K1DI1
9,K1DI1-101-1041-10,1-201-2011-76,K1DI1
1192639,K1DI1-101-1041-10,2-202-2022-219,K1DI1


,ID_K1BE2,Part_ID,Engine Type
0,K1BE2-101-1011-1,1-202-2021-339,K1BE2
409422,K1BE2-101-1011-1,2-202-2022-171,K1BE2
9,K1BE2-101-1011-10,1-202-2021-71,K1BE2
409431,K1BE2-101-1011-10,2-202-2022-145,K1BE2
435,K1BE2-101-1011-100,1-201-2011-1379,K1BE2


,ID_K1DI2,Part_ID,Engine Type
56,K1DI2-102-1021-1,1-201-2011-352,K1DI2
409478,K1DI2-102-1021-1,2-202-2022-1051,K1DI2
65,K1DI2-102-1021-10,1-202-2021-597,K1DI2
409487,K1DI2-102-1021-10,2-202-2022-1400,K1DI2
211,K1DI2-102-1021-100,1-202-2021-1049,K1DI2


Part_Type                0
Part_ID             640820
Herstellernummer    640820
Produktionsdatum    640820
Fehlerhaft          640820
dtype: int64

ID_K1BE1       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1DI1       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1BE2       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1DI2       0
Part_ID        0
Engine Type    0
dtype: int64

Merging the df

In [26]:
#K1BE1
merged_k1be1 = pd.merge(einzelteile_zusammen, melted_k1be1, how='inner', on="Part_ID")
merged_k1be1 = merged_k1be1.sort_values(["ID_K1BE1", "Part_Type"])
display(merged_k1be1.head())
display(merged_k1be1.isna().sum())

#K1DI1
merged_k1di1 = pd.merge(einzelteile_zusammen, melted_k1di1, how='inner', on="Part_ID")
merged_k1di1 = merged_k1di1.sort_values(["ID_K1DI1", "Part_Type"])
display(merged_k1di1.head())
display(merged_k1di1.isna().sum())

#K1BE2
merged_k1be2 = pd.merge(einzelteile_zusammen, melted_k1be2, how='inner', on="Part_ID")
merged_k1be2 = merged_k1be2.sort_values(["ID_K1BE2", "Part_Type"])
display(merged_k1be2.head())
display(merged_k1be2.isna().sum())

#K1DI2
merged_k1di2 = pd.merge(einzelteile_zusammen, melted_k1di2, how='inner', on="Part_ID")
merged_k1di2 = merged_k1di2.sort_values(["ID_K1DI2", "Part_Type"])
display(merged_k1di2.head())
display(merged_k1di2.isna().sum())


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1BE1,Engine Type
26,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
954338,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2147032,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
3339662,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
39,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1BE1            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1DI1,Engine Type
954593,T02,2-201-2011-144,201,2008-11-07,0,K1DI1-101-1041-1,K1DI1
2743559,T05,5-202-2012-89,202,2008-11-07,0,K1DI1-101-1041-1,K1DI1
48,T01,1-201-2011-76,201,2008-11-07,0,K1DI1-101-1041-10,K1DI1
1312100,T02,2-202-2022-219,202,2008-11-07,0,K1DI1-101-1041-10,K1DI1
2743594,T05,5-202-2012-14,202,2008-11-07,0,K1DI1-101-1041-10,K1DI1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1DI1            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1BE2,Engine Type
163668,T01,1-202-2021-339,202,2008-11-07,1,K1BE2-101-1011-1,K1BE2
450514,T02,2-202-2022-171,202,2008-11-07,0,K1BE2-101-1011-1,K1BE2
163652,T01,1-202-2021-71,202,2008-11-07,0,K1BE2-101-1011-10,K1BE2
450117,T02,2-202-2022-145,202,2008-11-07,1,K1BE2-101-1011-10,K1BE2
192,T01,1-201-2011-1379,201,2008-11-10,0,K1BE2-101-1011-100,K1BE2


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1BE2            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1DI2,Engine Type
8,T01,1-201-2011-352,201,2008-11-07,0,K1DI2-102-1021-1,K1DI2
450080,T02,2-202-2022-1051,202,2008-11-08,0,K1DI2-102-1021-1,K1DI2
163221,T01,1-202-2021-597,202,2008-11-08,0,K1DI2-102-1021-10,K1DI2
450086,T02,2-202-2022-1400,202,2008-11-08,1,K1DI2-102-1021-10,K1DI2
163278,T01,1-202-2021-1049,202,2008-11-09,0,K1DI2-102-1021-100,K1DI2


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1DI2            0
Engine Type         0
dtype: int64

Checking if motor is in OM1

In [27]:
# Importing OEM1 data
oem11 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")
oem12 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")

# Combining the OEM1 Types
oem_combined = pd.concat([oem11, oem12], ignore_index=True)

#using only the needed column
oem_combined = oem_combined["ID_Motor"]

display(oem_combined.head())
display(oem_combined.isna().sum())

0     K1BE1-101-1011-7
1    K1BE1-101-1011-12
2    K1BE1-101-1011-38
3    K1BE1-101-1011-97
4    K1BE1-101-1011-65
Name: ID_Motor, dtype: str

np.int64(0)

Combining all merged df into one

In [28]:
#renaming the columns before the concat
merged_k1be1 = merged_k1be1.rename(columns={"ID_K1BE1": "Motor_ID"})
merged_k1di1 = merged_k1di1.rename(columns={"ID_K1DI1": "Motor_ID"})
merged_k1be2 = merged_k1be2.rename(columns={"ID_K1BE2": "Motor_ID"})
merged_k1di2 = merged_k1di2.rename(columns={"ID_K1DI2": "Motor_ID"})

display(merged_k1be1.head(), merged_k1di1.head(), merged_k1be2.head(), merged_k1di2.head())

final_df = pd.concat([merged_k1be1, merged_k1di1, merged_k1be2, merged_k1di2], ignore_index=True)
display(final_df.head(), final_df.tail())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
26,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
954338,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2147032,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
3339662,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
39,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
954593,T02,2-201-2011-144,201,2008-11-07,0,K1DI1-101-1041-1,K1DI1
2743559,T05,5-202-2012-89,202,2008-11-07,0,K1DI1-101-1041-1,K1DI1
48,T01,1-201-2011-76,201,2008-11-07,0,K1DI1-101-1041-10,K1DI1
1312100,T02,2-202-2022-219,202,2008-11-07,0,K1DI1-101-1041-10,K1DI1
2743594,T05,5-202-2012-14,202,2008-11-07,0,K1DI1-101-1041-10,K1DI1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
163668,T01,1-202-2021-339,202,2008-11-07,1,K1BE2-101-1011-1,K1BE2
450514,T02,2-202-2022-171,202,2008-11-07,0,K1BE2-101-1011-1,K1BE2
163652,T01,1-202-2021-71,202,2008-11-07,0,K1BE2-101-1011-10,K1BE2
450117,T02,2-202-2022-145,202,2008-11-07,1,K1BE2-101-1011-10,K1BE2
192,T01,1-201-2011-1379,201,2008-11-10,0,K1BE2-101-1011-100,K1BE2


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
8,T01,1-201-2011-352,201,2008-11-07,0,K1DI2-102-1021-1,K1DI2
450080,T02,2-202-2022-1051,202,2008-11-08,0,K1DI2-102-1021-1,K1DI2
163221,T01,1-202-2021-597,202,2008-11-08,0,K1DI2-102-1021-10,K1DI2
450086,T02,2-202-2022-1400,202,2008-11-08,1,K1DI2-102-1021-10,K1DI2
163278,T01,1-202-2021-1049,202,2008-11-09,0,K1DI2-102-1021-100,K1DI2


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
9345273,T02,2-201-2011-593523,201,2013-10-16,0,K1DI2-103-1031-99997,K1DI2
9345274,T01,1-201-2011-791596,201,2013-10-19,0,K1DI2-103-1031-99998,K1DI2
9345275,T02,2-202-2022-1384696,202,2013-10-17,0,K1DI2-103-1031-99998,K1DI2
9345276,T01,1-201-2011-790687,201,2013-10-17,1,K1DI2-103-1031-99999,K1DI2
9345277,T02,2-202-2022-1384502,202,2013-10-17,0,K1DI2-103-1031-99999,K1DI2


looking if the engine is in oem1

In [29]:
final_df["in_ome1"] = final_df["Motor_ID"].isin(oem_combined).astype(int)


display(final_df.head())
display(final_df.isna().sum())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type,in_ome1
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1,1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1,1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
Motor_ID            0
Engine Type         0
in_ome1             0
dtype: int64

renaming the columns

In [30]:
final_df = final_df.rename(columns={
    "Part_Type": "part_type",
    "Part_ID" : "part_id",
    "Herstellernummer" : "manufacturer",
    "Produktionsdatum" : "production_date",
    "Fehlerhaft" : "faulty",
    "Motor_ID" : "engine_id",
    "Engine Type" : "engine_type",
    "in_ome1" : "OEM_type"
                            })

display(final_df.head())

,part_type,part_id,manufacturer,production_date,faulty,engine_id,engine_type,OEM_type
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1,1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1,1


Exporting to csv

In [31]:
final_df.to_csv("final_df.csv", index=False)